# MIDI 清理：删除表情参数，只保留音符和 meta，并统一 Channel 1

目的：

```text
输入：
D:\organ-amt-generalization\data\raw\organ\BWV545\midi

输出：
D:\organ-amt-generalization\data\processed\clearmidi_noexpr
```

清理规则：

```text
保留：
- note_on
- note_off
- meta events：tempo / time_signature / key_signature / track_name / end_of_track 等

修改：
- 所有 note_on / note_off → Channel 1

删除：
- control_change
- program_change
- pitchwheel
- aftertouch
- polytouch
- sysex
- 其他非 note 的 channel 事件
```

重要：删除事件时，脚本会保留 delta time，不会压缩 MIDI 时间轴。

In [1]:
# 如果当前环境没装依赖，先运行
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "mido", "pyyaml", "pandas"
])

0

In [2]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
import mido
from collections import Counter, defaultdict

## 1. 路径配置

确认项目中有：

```text
D:\organ-amt-generalization\scripts\clean_midi_no_expr.py
```

In [3]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\organ-amt-generalization")

SCRIPT_PATH = Path(r"D:\organ-amt-generalization\scripts\clean_midi_no_expr.py")
CONFIG_PATH = Path(r"D:\organ-amt-generalization\configs\clean_midi_no_expr.yaml")

INPUT_MIDI_DIR = Path(r"D:\organ-amt-generalization\data\raw\organ\BWV545\midi")
OUTPUT_MIDI_DIR = Path(r"D:\organ-amt-generalization\data\processed\clearmidi_noexpr")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SCRIPT_PATH:", SCRIPT_PATH)
print("CONFIG_PATH:", CONFIG_PATH)
print("INPUT_MIDI_DIR:", INPUT_MIDI_DIR)
print("OUTPUT_MIDI_DIR:", OUTPUT_MIDI_DIR)

assert PROJECT_ROOT.exists(), f"项目根目录不存在: {PROJECT_ROOT}"
assert SCRIPT_PATH.exists(), f"清理脚本不存在，请先放到: {SCRIPT_PATH}"
assert INPUT_MIDI_DIR.exists(), f"输入 MIDI 文件夹不存在: {INPUT_MIDI_DIR}"

PROJECT_ROOT: D:\organ-amt-generalization
SCRIPT_PATH: D:\organ-amt-generalization\scripts\clean_midi_no_expr.py
CONFIG_PATH: D:\organ-amt-generalization\configs\clean_midi_no_expr.yaml
INPUT_MIDI_DIR: D:\organ-amt-generalization\data\raw\organ\BWV545\midi
OUTPUT_MIDI_DIR: D:\organ-amt-generalization\data\processed\clearmidi_noexpr


## 2. 写入配置文件

配置文件写到：

```text
D:\organ-amt-generalization\configs\clean_midi_no_expr.yaml
```

In [4]:
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_MIDI_DIR.mkdir(parents=True, exist_ok=True)

config_text = f'''# clean_midi_no_expr.yaml

input_midi_dir: "{INPUT_MIDI_DIR.as_posix()}"
output_midi_dir: "{OUTPUT_MIDI_DIR.as_posix()}"

target_channel: 1

recursive: false
preserve_subfolders: true
overwrite: true

keep_meta_events: true
remove_sysex: true
remove_unknown_channel_events: true

write_metadata: true
metadata_filename: "clean_midi_no_expr_metadata.csv"
'''

CONFIG_PATH.write_text(config_text, encoding="utf-8")

print(CONFIG_PATH)
print(CONFIG_PATH.read_text(encoding="utf-8"))

D:\organ-amt-generalization\configs\clean_midi_no_expr.yaml
# clean_midi_no_expr.yaml

input_midi_dir: "D:/organ-amt-generalization/data/raw/organ/BWV545/midi"
output_midi_dir: "D:/organ-amt-generalization/data/processed/clearmidi_noexpr"

target_channel: 1

recursive: false
preserve_subfolders: true
overwrite: true

keep_meta_events: true
remove_sysex: true
remove_unknown_channel_events: true

write_metadata: true
metadata_filename: "clean_midi_no_expr_metadata.csv"



## 3. 清理前检查：channel 和控制事件

这一步只检查，不修改文件。

In [5]:
def inspect_midi(midi_path: Path) -> dict:
    mid = mido.MidiFile(str(midi_path))

    note_channel_count = Counter()
    channel_msg_count = Counter()
    event_type_count = Counter()
    control_count = Counter()

    for track in mid.tracks:
        for msg in track:
            event_type_count[msg.type] += 1

            if hasattr(msg, "channel"):
                channel_msg_count[msg.channel + 1] += 1

            if msg.type in {"note_on", "note_off"} and hasattr(msg, "channel"):
                note_channel_count[msg.channel + 1] += 1

            if msg.type == "control_change":
                control_count[msg.control] += 1

    return {
        "file": midi_path.name,
        "num_tracks": len(mid.tracks),
        "length_sec": getattr(mid, "length", None),
        "note_channel_count": dict(sorted(note_channel_count.items())),
        "channel_msg_count": dict(sorted(channel_msg_count.items())),
        "event_type_count": dict(sorted(event_type_count.items())),
        "control_count": dict(sorted(control_count.items())),
    }


input_midi_files = sorted(list(INPUT_MIDI_DIR.glob("*.mid")) + list(INPUT_MIDI_DIR.glob("*.midi")))
print("input midi files:", len(input_midi_files))

df_before = pd.DataFrame([inspect_midi(p) for p in input_midi_files])
display(df_before)

input midi files: 2


,file,num_tracks,length_sec,note_channel_count,channel_msg_count,event_type_count,control_count
0,BWV545i.mid,4,115.007579,"{1: 1102, 2: 570, 3: 340}","{1: 1108, 2: 575, 3: 344}","{'control_change': 12, 'copyright': 1, 'end_of...","{0: 3, 7: 4, 10: 2, 91: 3}"
1,BWV545ii.mid,5,229.026299,"{1: 894, 2: 864, 3: 990, 4: 508}","{1: 900, 2: 869, 3: 995, 4: 512}","{'control_change': 16, 'copyright': 1, 'end_of...","{0: 4, 7: 5, 10: 3, 91: 4}"


## 4. 运行清理脚本

In [6]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--config",
    str(CONFIG_PATH),
]

print("Running command:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    cwd=str(PROJECT_ROOT),
    text=True,
    capture_output=True,
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Script failed with return code {result.returncode}")

Running command:
d:\anaconda3\envs\m1\python.exe D:\organ-amt-generalization\scripts\clean_midi_no_expr.py --config D:\organ-amt-generalization\configs\clean_midi_no_expr.yaml
STDOUT:
[INFO] Input MIDI dir: D:\organ-amt-generalization\data\raw\organ\BWV545\midi
[INFO] Output MIDI dir: D:\organ-amt-generalization\data\processed\clearmidi_noexpr
[INFO] Found MIDI files: 2
[INFO] Target MIDI channel: 1
[INFO] Recursive: False
[INFO] Preserve subfolders: True
[INFO] Overwrite: True
[INFO] Cleaning rule: keep notes + meta, remove CC / program_change / pitchwheel / aftertouch / sysex.

[1/2] BWV545i.mid
[OK] Saved: D:\organ-amt-generalization\data\processed\clearmidi_noexpr\BWV545i.mid
[INFO] notes=2012, removed=15, overlap_note_on=2, orphan_note_off=0

[2/2] BWV545ii.mid
[OK] Saved: D:\organ-amt-generalization\data\processed\clearmidi_noexpr\BWV545ii.mid
[INFO] notes=3256, removed=20, overlap_note_on=8, orphan_note_off=0

[DONE] Processed: 2
[DONE] Output files recorded: 2
[DONE] Output dir

## 5. 清理后检查输出文件

In [7]:
output_midi_files = sorted(list(OUTPUT_MIDI_DIR.glob("*.mid")) + list(OUTPUT_MIDI_DIR.glob("*.midi")))
print("output midi files:", len(output_midi_files))

for p in output_midi_files:
    print(p)

output midi files: 2
D:\organ-amt-generalization\data\processed\clearmidi_noexpr\BWV545i.mid
D:\organ-amt-generalization\data\processed\clearmidi_noexpr\BWV545ii.mid


## 6. 验证：是否只剩 note + meta，且 note 全部为 Channel 1

In [8]:
df_after = pd.DataFrame([inspect_midi(p) for p in output_midi_files])
display(df_after)

,file,num_tracks,length_sec,note_channel_count,channel_msg_count,event_type_count,control_count
0,BWV545i.mid,4,115.007579,{1: 2012},{1: 2012},"{'copyright': 1, 'end_of_track': 4, 'note_off'...",{}
1,BWV545ii.mid,5,229.026299,{1: 3256},{1: 3256},"{'copyright': 1, 'end_of_track': 5, 'note_off'...",{}


In [9]:
def check_clean_result(midi_path: Path) -> dict:
    mid = mido.MidiFile(str(midi_path))

    bad_events = []
    note_channels = Counter()

    for ti, track in enumerate(mid.tracks):
        for msg in track:
            if msg.is_meta:
                continue

            if msg.type in {"note_on", "note_off"}:
                note_channels[msg.channel + 1] += 1
            else:
                bad_events.append((ti, msg.type))

    return {
        "file": midi_path.name,
        "note_channels": dict(sorted(note_channels.items())),
        "only_note_channel_1": set(note_channels.keys()) <= {1},
        "num_bad_non_note_events": len(bad_events),
        "bad_event_types": dict(Counter([x[1] for x in bad_events])),
    }


df_check = pd.DataFrame([check_clean_result(p) for p in output_midi_files])
display(df_check)

,file,note_channels,only_note_channel_1,num_bad_non_note_events,bad_event_types
0,BWV545i.mid,{1: 2012},True,0,{}
1,BWV545ii.mid,{1: 3256},True,0,{}


## 7. 读取 metadata

重点看：

```text
num_messages_removed
removed_event_counts_json
overlap_note_on_count
orphan_note_off_count
```

如果 `overlap_note_on_count` 很大，说明强制 Channel 1 后同音高重叠较多，后续可能需要用 `pretty_midi + add_midi_note` 方式渲染。

In [10]:
metadata_csv = OUTPUT_MIDI_DIR / "clean_midi_no_expr_metadata.csv"

if metadata_csv.exists():
    df_meta = pd.read_csv(metadata_csv)
    display(df_meta)
else:
    print("metadata csv not found:", metadata_csv)

,time,input_midi_path,output_midi_path,target_channel,num_tracks,num_note_messages_kept,num_meta_messages_kept,num_messages_removed,removed_event_counts_json,overlap_note_on_count,orphan_note_off_count,status,error_message
0,2026-05-08T23:26:08,D:\organ-amt-generalization\data\raw\organ\BWV...,D:\organ-amt-generalization\data\processed\cle...,1,4,2012,27,15,"{""control_change"": 12, ""program_change"": 3}",2,0,ok,NaN
1,2026-05-08T23:26:08,D:\organ-amt-generalization\data\raw\organ\BWV...,D:\organ-amt-generalization\data\processed\cle...,1,5,3256,31,20,"{""control_change"": 16, ""program_change"": 4}",8,0,ok,NaN


## 8. 检查指定时间窗口的 note 事件

把 `MIDI_PATH`、`START_SEC`、`END_SEC` 改成你怀疑高音缺失的文件和时间段。

In [11]:
MIDI_PATH = output_midi_files[0] if output_midi_files else None

START_SEC = 85.0
END_SEC = 90.0

def midi_note_events_with_seconds(midi_path: Path):
    mid = mido.MidiFile(str(midi_path))
    tempo = 500000
    rows = []

    merged = mido.merge_tracks(mid.tracks)
    abs_sec = 0.0

    for msg in merged:
        abs_sec += mido.tick2second(msg.time, mid.ticks_per_beat, tempo)

        if msg.is_meta:
            if msg.type == "set_tempo":
                tempo = msg.tempo
            continue

        if msg.type in {"note_on", "note_off"}:
            rows.append({
                "time_sec": abs_sec,
                "type": msg.type,
                "note": msg.note,
                "velocity": getattr(msg, "velocity", None),
                "channel": getattr(msg, "channel", None) + 1,
                "is_note_off_like": (
                    msg.type == "note_off" or 
                    (msg.type == "note_on" and getattr(msg, "velocity", 0) == 0)
                )
            })

    return pd.DataFrame(rows)


if MIDI_PATH is not None:
    df_events = midi_note_events_with_seconds(MIDI_PATH)

    df_win = df_events[
        (df_events["time_sec"] >= START_SEC) &
        (df_events["time_sec"] <= END_SEC)
    ].copy()

    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)

    df_win_sorted = df_win.sort_values(
        ["note", "time_sec"],
        ascending=[False, True]
    ).reset_index(drop=True)

    print("MIDI_PATH:", MIDI_PATH)
    print("window:", START_SEC, END_SEC)
    print("min/max time in window:", df_win_sorted["time_sec"].min(), df_win_sorted["time_sec"].max())

    display(df_win_sorted)
else:
    print("No output MIDI files.")

MIDI_PATH: D:\organ-amt-generalization\data\processed\clearmidi_noexpr\BWV545i.mid
window: 85.0 90.0
min/max time in window: 85.05779612500038 89.99048349999997


,time_sec,type,note,velocity,channel,is_note_off_like
0,85.067412,note_on,74,100,1,False
1,85.134719,note_off,74,100,1,True
2,85.221258,note_on,74,100,1,False
3,85.288565,note_off,74,100,1,True
4,85.375104,note_on,74,100,1,False
5,85.442411,note_off,74,100,1,True
6,85.528950,note_on,74,100,1,False
7,85.596257,note_off,74,100,1,True
8,85.682796,note_on,74,100,1,False
9,85.750103,note_off,74,100,1,True


## 9. 下一步渲染

后续 DAWDreamer 渲染时，把 MIDI 输入文件夹换成：

```text
D:\organ-amt-generalization\data\processed\clearmidi_noexpr
```

如果这个版本高音恢复，说明主要问题来自 MIDI 表情参数或 program change。

如果仍然缺高音，再优先尝试 `pretty_midi + add_midi_note` 渲染，绕开 MIDI 原始 note_off 事件流。